# 22 — Held-out READ Go/No-Go decision

This notebook is model-free. It computes the preregistered held-out
concept-level AUCs and the single GO/NO-GO decision, then stops.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

from src.read_validation import (
    CANDIDATE_NAMES,
    READ_VALIDATION_PROTOCOL,
    READ_VALIDATION_PROTOCOL_SHA256,
    build_model_free_validation_report,
    read_json,
    write_json,
)

ROOT = Path.cwd()
RAW_DIR = ROOT / 'data/raw/v5'
raw20 = read_json(RAW_DIR / '20_read_ground_truth.json')
raw21 = read_json(RAW_DIR / '21_read_estimators.json')
assert raw20['protocol_sha256'] == raw21['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
scored_rows = raw21['rows']
assert len(scored_rows) == 163
validation = build_model_free_validation_report(scored_rows)
decision = validation['go_no_go']
print(decision['one_line'])
print(json.dumps({candidate: validation['pooled_oof_auc']['candidate_results'][candidate] for candidate in CANDIDATE_NAMES}, indent=2))

In [ ]:
fig_dir = ROOT / 'results/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
auc_results = validation['pooled_oof_auc']['candidate_results']
x = np.arange(len(CANDIDATE_NAMES))
auc_values = [auc_results[name].get('auc', np.nan) if auc_results[name].get('auc') is not None else np.nan for name in CANDIDATE_NAMES]
low = [auc_results[name].get('ci95_low', np.nan) if auc_results[name].get('ci95_low') is not None else np.nan for name in CANDIDATE_NAMES]
high = [auc_results[name].get('ci95_high', np.nan) if auc_results[name].get('ci95_high') is not None else np.nan for name in CANDIDATE_NAMES]
yerr = np.asarray([[a-b if np.isfinite(a) and np.isfinite(b) else 0 for a,b in zip(auc_values,low)], [b-a if np.isfinite(a) and np.isfinite(b) else 0 for a,b in zip(auc_values,high)]])

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(x, np.nan_to_num(auc_values, nan=0.0), color=['#6a1b9a'] + ['#00897b']*3 + ['#1565c0']*3)
finite = np.isfinite(auc_values)
if finite.any():
    ax.errorbar(x[finite], np.asarray(auc_values)[finite], yerr=yerr[:, finite], fmt='none', color='black', capsize=4)
for index, (bar, value) in enumerate(zip(bars, auc_values)):
    if np.isfinite(value):
        ax.text(bar.get_x()+bar.get_width()/2, value+0.025, f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    else:
        ax.text(bar.get_x()+bar.get_width()/2, 0.03, 'NOT\nESTIMABLE', ha='center', va='bottom', fontsize=8, color='red')
ax.axhline(0.70, color='red', linestyle='--', label='GO point bar 0.70')
ax.axhline(0.50, color='black', linestyle=':', label='chance / CI-lower bar')
ax.set_xticks(x, CANDIDATE_NAMES, rotation=30, ha='right')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Pooled out-of-fold ROC AUC')
ax.set_title('F1 — READ engine-vs-dashboard validation (79 semantic concepts)')
ax.legend()
fig.tight_layout()
f1 = fig_dir / 'f1_read_go_no_go_auc.png'
fig.savefig(f1, dpi=180)
plt.close(fig)

behavior_candidates = [name for name in CANDIDATE_NAMES if name.startswith(('R2_', 'R3_'))]
estimable = [name for name in behavior_candidates if auc_results[name]['status'] == 'OK']
best = max(estimable, key=lambda name: auc_results[name]['auc']) if estimable else 'R2_sum'
concept_rows = validation['calibration']['concept_rows']
fig, ax = plt.subplots(figsize=(9, 7))
for class_name, color, marker in [('engine', '#1565c0', 'o'), ('dashboard', '#ef6c00', 's')]:
    subset = [row for row in concept_rows if row['class_name'] == class_name and row['A_causal_abs'] is not None and row['candidate_scores'][best] is not None]
    ax.scatter([row['A_causal_abs'] for row in subset], [row['candidate_scores'][best] for row in subset], c=color, marker=marker, alpha=0.8, label=f'{class_name} (N={len(subset)})')
ax.set_xlabel('|CAUSAL(A)| coordinate-resampling effect')
ax.set_ylabel(f'OOF train-CDF score: {best}')
best_auc = auc_results[best].get('auc')
best_ci = (auc_results[best].get('ci95_low'), auc_results[best].get('ci95_high'))
ax.set_title(f'F2 — best behavior-specific READ vs patching A\n{best}: AUC={best_auc}, 95% CI={best_ci}')
ax.legend()
fig.tight_layout()
f2 = fig_dir / 'f2_best_read_vs_patching.png'
fig.savefig(f2, dpi=180)
plt.close(fig)
print({'F1': str(f1), 'F2': str(f2), 'best_behavior_specific': best})

In [ ]:
raw22 = {
    'schema_version': 'read-go-no-go-decision-v1',
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'validation': validation,
    'figures': [str(f1.relative_to(ROOT)), str(f2.relative_to(ROOT))],
}
raw22_artifact = write_json(RAW_DIR / '22_read_decision.json', raw22)

metrics_path = ROOT / 'results/metrics.json'
metrics = read_json(metrics_path)
v5 = metrics['read_validation_v5']
assert v5['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
v5['stage22'] = {
    'status': 'COMPLETE',
    'raw_artifact': raw22_artifact,
    'figures': raw22['figures'],
    'validation': validation,
}
v5['status'] = 'COMPLETE'
v5['decision'] = decision['decision']
v5['decision_one_line'] = decision['one_line']
v5['hypothesis_status'] = 'NOT_TESTED'
write_json(metrics_path, metrics)

auc_lines = []
for name in CANDIDATE_NAMES:
    result = auc_results[name]
    auc_lines.append(
        f"| {name} | {result.get('status')} | {result.get('auc')} | "
        f"[{result.get('ci95_low')}, {result.get('ci95_high')}] | "
        f"{result.get('complete_concept_coverage')} | {result.get('passes_go_threshold')} |"
    )
correlation = validation['secondary_correlations']['A_B_correlation']
labels = validation['label_verification']
counts = validation['calibration']
report = f'''# READ Go/No-Go validation

## DECISION

**{decision['one_line']}**

This is a READ-method validation result only. Written-vs-Read P1/P2/P3 remain **NOT TESTED**.

## Preflight and model scope

- GPU: {v5['preflight']['gpu']['name']}; {v5['preflight']['gpu']['memory_total_mib']} MiB total, {v5['preflight']['gpu']['memory_free_mib']} MiB free.
- HF filesystem free: {v5['preflight']['disk']['free_gib']} GiB.
- Model: `Qwen/Qwen2.5-7B-Instruct`, bf16.
- Qwen3 scale arm: **SKIPPED** because no comparable validated Qwen3 J-Lens/direction instrument exists; 32B/30B also exceed disk and 4-bit cannot run the derivative protocol.

## Ground truth

Primary A is concept-coordinate resampling from a frozen clean donor. It is independent of READ weights/gradients but depends on the frozen J-Lens direction; it is not full-residual patching. Secondary B is the fixed masked source-to-foil clamped swap at alpha=1.5; it is not a source-only deletion.

A/B concept-level correlation: Spearman `{correlation['spearman'].get('rho')}`; Pearson `{correlation['pearson'].get('r')}`; N={correlation['spearman'].get('n_concepts')}.

Declared-label verification: **{labels['status']}**; failures={labels['n_failures']}/163. Failures were retained. Clean capability was diagnostic only.

## Held-out classification

Inference uses {counts['n_concepts']} semantic concepts ({counts['concept_counts_by_class']['engine']} engines, {counts['concept_counts_by_class']['dashboard']} dashboard languages), not 163 independent concepts. Four grouped folds keep source/foil connected components and each language together. R2/R3 paths and R2 position profiles use training folds only; OOF scores use unlabeled training-concept CDF calibration.

| candidate | status | OOF AUC | cluster-bootstrap 95% CI | complete coverage | GO bar |
| --- | --- | ---: | --- | --- | --- |
{chr(10).join(auc_lines)}

The requested A-vs-B AUC rows are numerically identical because the fixed engine/dashboard labels—not causal magnitudes—define ROC AUC. They are one inferential look, not two chances to pass. Spearman associations against continuous A and B are retained in `metrics.json`.

![F1](figures/f1_read_go_no_go_auc.png)

![F2](figures/f2_best_read_vs_patching.png)

## Estimators

- R1: global repaired-weight READ across every MLP/head in the compute-bounded common L25–27 frontier.
- R2: fixed train-fold top-8 path (2 MLP + 6 heads), with static sum and train-derived relative-position peak/carrying summaries.
- R3: exact input-dependent real-activation derivative on that same train-fold path, with position-normalized sum, peak, and carrying mean.

No estimator beyond R1/R2/R3 was added; no alpha sweep, controls, nulls, capability run, ambiguity, scale science, P1/P2/P3, tests, or lint were run.

## Limitations

- Only four independent dashboard concepts exist; the 10,000-draw CI resamples dependency clusters and is intrinsically fragile.
- Engine and dashboard labels are task/domain-confounded (two-hop facts versus multilingual continuation).
- The roster is reused calibration data, not an untouched external holdout.
- A uses direction-coordinate rather than full-state resampling because the repository has no matched clean/corrupt prompt pairs.
- Frozen donor activations are an exogenous corruption bank and may cross evaluation folds; donor folds are never used as labels or path-selection observations, but dependence remains.
- L25–27 is a compute-bounded common frontier, not all downstream components.

## Reproducibility

- Protocol SHA-256: `{READ_VALIDATION_PROTOCOL_SHA256}`.
- Notebook-20 raw: `{v5['stage20']['raw_artifact']['path']}` (`{v5['stage20']['raw_artifact']['sha256']}`).
- Notebook-21 raw: `{v5['stage21']['raw_artifact']['path']}` (`{v5['stage21']['raw_artifact']['sha256']}`).
- Notebook-22 raw: `{raw22_artifact['path']}` (`{raw22_artifact['sha256']}`).
'''
(ROOT / 'results/RESULTS.md').write_text(report, encoding='utf-8')
print(report)
assert decision['decision'] in {'GO', 'NO-GO'}
assert 'P1/P2/P3 remain **NOT TESTED**' in report
assert len(CANDIDATE_NAMES) == 7
print(decision['one_line'])